# 02 — Hazard labels and exploratory analysis (Phase 4)

Builds the heatwave and extreme-wind labels with `climax_hazards.labels` and shows prevalence, threshold sensitivity,
seasonality and spatial distribution. CPU only, a few minutes.

* All data-derived thresholds come from the **train split (1979–2010)** only and are applied unchanged to val/test.
* Labels: 1 = event, 0 = no event, NaN = undetermined (missing day, masked cell, or a spell cut by a gap or split boundary).
* Definitions and rationale: `docs/mtech/HAZARD_DEFINITIONS.md`.
* Extreme precipitation needs IMD 0.25° gridded rainfall (`rain/<year>.grd`, 1979–2018) attached as a Kaggle dataset; it is skipped if none is found.

Attach the regional dataset. Use the corrected 6×6 dataset (lon 67.5–95.625°E) if you uploaded it; otherwise the original 6×5 one works too.

In [ ]:
INPUT_ROOT = "/kaggle/input"
DATA_DIR_OVERRIDE = None     # set if several datasets are attached
IMD_DIR_OVERRIDE = None      # folder that contains rain/1979.grd ... rain/2018.grd
REPO_DIR_OVERRIDE = None
REPO_URL = "https://github.com/Aditya1Agrawal/ClimaX-Monsoon-PINN.git"
REPO_BRANCH = "mtech-extreme-events"
OUT_DIR = "/kaggle/working/labels"
RUN_SENSITIVITY = True

In [ ]:
import os, sys, json, subprocess
def find_repo(root):
    for dirpath, _, _ in os.walk(root):
        if os.path.isdir(os.path.join(dirpath, "src", "climax_hazards")):
            return dirpath
REPO_DIR = REPO_DIR_OVERRIDE or find_repo(INPUT_ROOT)
if REPO_DIR is None:
    REPO_DIR = "/kaggle/working/ClimaX-Monsoon-PINN"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
sys.path.insert(0, os.path.join(REPO_DIR, "scripts", "mtech"))
print(subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

from climax_hazards import inspect_utils as iu, labels as L
cands = [p for p in iu.find_dataset_dirs(INPUT_ROOT) if "fake_era5_demo" not in str(p)]
DATA_DIR = str(iu.resolve_single(cands, DATA_DIR_OVERRIDE, "regional dataset"))
pad = iu.detect_lon_padding(DATA_DIR)
print("DATA_DIR:", DATA_DIR, "| lon:", pad["lon"], "| padded:", pad["looks_padded"])
assert not pad["looks_padded"], "use an unpadded dataset; a duplicated column would double-count events"

def find_imd(root):
    for dirpath, dirnames, filenames in os.walk(root):
        if os.path.basename(dirpath) == "rain" and "1979.grd" in filenames:
            return os.path.dirname(dirpath)
IMD_DIR = IMD_DIR_OVERRIDE or find_imd(INPUT_ROOT)
print("IMD_DIR:", IMD_DIR or "not found - precipitation labels skipped")

## Primary labels

In [ ]:
import numpy as np, pandas as pd
cache = {}
hw = L.heatwave_labels(DATA_DIR, L.HeatwaveConfig(), cache=cache)
wind = L.wind_labels(DATA_DIR, L.WindConfig(), cache=cache)
primary = [hw["heatwave_pct"], hw["heatwave_imd"], wind]
if IMD_DIR:
    from climax_hazards import precip as P
    primary.append(P.precip_labels(DATA_DIR, IMD_DIR, P.PrecipConfig(), cache=cache))
meta = {ls.name: L.save_labelset(ls, OUT_DIR) for ls in primary}
rows = [{"label": n, "split": s, **v} for n, m in meta.items() for s, v in m["prevalence"].items()]
print(pd.DataFrame(rows).to_string(index=False))
print("\nconfigs:", json.dumps({n: m["config"] for n, m in meta.items()}, indent=1))

## Threshold sensitivity, seasonality, maps

In [ ]:
import make_labels as ML
os.makedirs(OUT_DIR, exist_ok=True)
if RUN_SENSITIVITY:
    sens = ML.sensitivity(DATA_DIR, cache)
    if IMD_DIR:
        sens = pd.concat([sens, ML.sensitivity_precip(DATA_DIR, IMD_DIR, cache)], ignore_index=True)
    sens.to_csv(os.path.join(OUT_DIR, "threshold_sensitivity.csv"), index=False)
    print("prevalence in % of valid cell-days")
    ML.print_sensitivity(sens)
ML.plots(primary, DATA_DIR, OUT_DIR)
from IPython.display import Image, display
display(Image(os.path.join(OUT_DIR, "label_maps_and_seasonality.png")))
display(Image(os.path.join(OUT_DIR, "label_annual_prevalence.png")))

## Forecast alignment check

A model input at hour `t` with lead 72 h is paired with the label of the UTC day containing `t + 72 h`, taken from the **same split**.
Targets that fall outside the split are NaN and are dropped, so no window crosses a split boundary.

In [ ]:
d = wind.splits["val"]
t_in = np.arange(np.datetime64("2014-12-27T00", "h"), np.datetime64("2015-01-01T00", "h"), 24)
aligned = L.align_to_forecast(t_in, 72, d["dates"], d["label"])
for t, a in zip(t_in, aligned):
    print(t, "-> target day", (t + np.timedelta64(72, "h")).astype("datetime64[D]"), "| valid cells:", int(np.isfinite(a).sum()))